<a href="https://colab.research.google.com/github/comp0161/tutorials/blob/main/lab06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP0161 Auditory Computing Week 6: Auditory Localisation

In this week's tutorial we'll look at some aspects of spatial sound and localisation, including simple stereo panning, reverberation and binaural synthesis.

You should be familiare with [Colab](https://colab.research.google.com/) by now, but if you need a refresher have a look at the [week 3 notebook](https://colab.research.google.com/github/comp0161/tutorials/blob/main/lab03.ipynb).

# Setting Up

Fetch some data from the module GitHub site that we'll make use of later.

In [ ]:
!git clone https://github.com/comp0161/labs_data.git data

Install some audio-specific libraries that aren't present by default on Colab VMs:

* [pyfar](https://pyfar.readthedocs.io/)
* [pedalboard](https://github.com/spotify/pedalboard)

<details>
<summary>Note</summary>
There's a fair amount of overlap between these libraries and I could probably streamline this notebook to use only one, but it's convenient to use both and doesn't really cost anything in this context.
</details>

In [ ]:
print('installing pyfar')
%pip install pyfar > /dev/null
print('done')

print('installing pedalboard')
%pip install pedalboard > /dev/null
print('done')

Import libraries and set up some global variables.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import Image, Audio

import pedalboard, pedalboard.io
import pyfar as pf

# we'll be wanting some random numbers later
# we create a generator with a fixed seed for consistent behaviour
# if you want different results, change the seed
SEED = 9907
rng = np.random.default_rng(SEED)

# Sounds

We're going to need some simple sounds for demonstration purposes. We could use pre-recorded clips, but instead we'll synthesise them from scratch.

The functions below are adapted from the [`lab01.py`](https://github.com/comp0161/tutorials/blob/main/lab01.py) script from tutorial 1. You don't need to understand them in any detail, but feel free to dig in if you want.

In [ ]:
PITCH = 440
PITCHES = [55, 110, 220, 440, 880]
DURATION = 0.5
SAMPLE_RATE = 44100

def play(x, rate=SAMPLE_RATE, dupe_stereo=True):
    """
    Display a numpy array using IPython.Audio.
    Optionally duplicates mono to stereo (on by default).
    """
    if dupe_stereo and (len(x.shape) == 1):
        x = np.stack((x,x)).copy()

    display(Audio(x, rate=rate))

def tone(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0):
    """
    Generate a simple pure sine tone of specified frequency
    and duration.
    """
    return np.sin(phase + hz * 2 * np.pi * np.arange(int(duration * rate))/rate)

def tones(hz=PITCHES, duration=DURATION, weights=None, rate=SAMPLE_RATE, phases=None):
    """
    Generate a (potentially weighted) sum of pure sine tones.
    """
    if weights is None:
        weights = np.ones(len(hz))

    if phases is None:
        phases = np.zeros(len(hz))

    result = weights[0] * tone(hz[0], duration, rate, phases[0])
    for ii in range(1, len(hz)):
        result += weights[ii] * tone(hz[ii], duration, rate, phases[ii])

    return result

def saw (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=0 ):
    """
    Generate a band-limited sawtooth wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by`random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate
    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = -np.sin(angles * hz)
    components = 1

    # add the harmonics
    for harm in range(2, max_harm + 1):
        result -= np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def square (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=0 ):
    """
    Generate a band-limited square wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by`random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate

    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = np.sin(angles * hz)

    components = 1

    # add the harmonics
    for harm in range(3, max_harm + 1, 2):
        result += np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def noise(duration=DURATION, rate=SAMPLE_RATE, shape=None):
    """
    Generate N noise samples with the power spectrum
    optionally shaped by the supplied function.
    """
    N = int(duration * SAMPLE_RATE)
    noise_spectrum = np.fft.rfft(rng.standard_normal(N))

    if shape is not None:
        shape_spectrum = shape(np.fft.rfftfreq(N))
        # normalise to preserve energy
        shape_spectrum /= np.sqrt(np.mean(shape_spectrum**2))

        noise_spectrum *= shape_spectrum

    return np.fft.irfft(noise_spectrum);

BLUE = lambda x: np.sqrt(x)
VIOLET = lambda x: x
BROWN = lambda x: 1/np.where(x==0, np.inf, x)
PINK = lambda x: 1/np.where(x==0, np.inf, np.sqrt(x))

def envelope(duration=DURATION, rate=SAMPLE_RATE,
             attack=0.01, decay=0.05, sustain=0, release=0):
    """
    Simple linear ADSR envelope. Attack, decay and sustain are
    specified in seconds, generated according to the rate. Sustain
    fills everything not taken up by the other phases. If the
    duration is insufficient to contain all the elements the
    excess is discarded.
    """
    N = int(duration * rate)
    result = np.full(N, sustain, dtype=float)

    nR = np.min((int(release * rate), N))
    if nR:
        result[(N - nR):] = np.linspace(sustain, 0, nR)

    nA = int(attack * rate)
    if nA:
        result[:nA] = np.linspace(0, 1, nA)

    nD = np.min((int(decay * rate), N-nA))
    if nD:
        result[nA:(nA + nD)] = np.linspace(1, result[np.min((N, nA+nD))], nD)

    return result

We use these functions to generate some very basic sound building blocks. Note that all these sounds have the same duration, half a second.

In [ ]:
KICK = noise(shape=PINK) * envelope() + saw(55) * envelope(decay=0.2)
TINK = noise(shape=BLUE) * envelope() + saw(4400) * envelope(decay=0.1)
BEEP = square(330) * envelope(attack=0.0, decay=0.2)
BUZZ = square(44) * envelope(attack=0.05, decay=0.2)
REST = np.zeros_like(KICK)

play(KICK)
play(TINK)
play(BEEP)
play(BUZZ)

We can combine these into some (again very rudimentary) loops by concatenating one after another.

In [ ]:
DRUMS = np.concatenate((KICK, REST, BUZZ, REST))
HATS = np.concatenate((TINK, TINK, TINK, TINK))
KICK_ONLY = np.concatenate((KICK, REST, REST, REST))
HATS_OFF = np.concatenate((REST, TINK, TINK, TINK))

play(DRUMS)
play(HATS)
play(KICK_ONLY)
play(HATS_OFF)

We can also combine these in parallel just by summation.

<details>
<summary>Note</summary>
`IPython.Audio` normalises the input data by default so it will always be in the right range. In other contexts we might need to scale after adding.
<details>

In [ ]:
play(DRUMS + HATS)
play(KICK_ONLY + HATS_OFF)

Note that this only works if the sounds are the same length. The following will produce an error because `DRUMS` is four times as long as `TINK`.

In [ ]:
play(DRUMS + TINK)

# Stereo

So far the sounds are purely monophonic and (on headphones) will be delivered identically to each ear. If we want to produce a sense of localisation in space, we must introduce differences between the sounds received on both sides.

As an extreme case, we could have completely different loops on the left and right:

In [ ]:
play(np.stack((KICK_ONLY, HATS_OFF)))

We can be a bit more subtle by having both sounds present on each side but at different levels.

In [ ]:
def mono(x):
  """
  Flatten a multichannel sound to mono by summing all channels.
  """
  assert(len(x.shape)==2)
  return np.sum(x, axis=0)


def stereo(x, pan=0.5):
  """
  Generate a stereo sound from mono by adjusting
  the level on each side.

  `pan` goes from 0 (fully left) to 1 (fully right).

  (Out of range pan values will also work but may produce strange effects.)
  """
  assert(len(x.shape)==1)
  return np.stack((x * 1-pan, x * pan))


In [ ]:
play(stereo(HATS, 0.25))
play(stereo(HATS_OFF, 0.25) + stereo(KICK_ONLY, 0.75))

By gradually shifting stereo positions we can produce a crude sense of the sounds changing position in space.

In [ ]:
PAN = np.concatenate([stereo(KICK_ONLY, left) + stereo(HATS_OFF, 1 - left) for left in np.linspace(0, 1, 8)], axis=1)
play(PAN)

Real sound localisation depends not just on level cues but also on timing cues. We can attempt to emulate this by playing the same sound on both channels but with a small delay on one side.

In [ ]:
def delay_stereo(x, offset=30):
  """
  Duplicate a mono sound to stereo with a delay on one channel.
  `offset` specifies the delay in samples. If it is negative,
  the right channel is delayed, if it is positive the left channel is.
  """
  assert(len(x.shape)==1)

  if offset < 0:
    return np.stack((x, np.concatenate((np.zeros(-offset), x[:offset]))))
  elif offset > 0:
    return np.stack((np.concatenate((np.zeros(offset), x[:-offset])),x))
  else:
    return np.stack((x,x))


In [ ]:
play(np.concatenate([delay_stereo(TINK, off) for off in range(-300, 301, 30)], axis=1))

This only really works for a small range of offsets — the ears are pretty close together and it doesn't take long to get from one to the other when travelling at the speed of sound. Much longer than that and the brain won't interpret the difference as spatial but instead start perceiving distinct sounds.

In [ ]:
play(delay_stereo(TINK, 1000))

Combining level and timing differences like these for improved spatialisation is left as an exercise for the reader. But we'll come back to it in the context of binaural synthesis below.

# Reverberation

As noted in week 4, sounds in enclosed spaces such as rooms don't reach our ears only by the most direct route. They also bounce off the surfaces in the room, and maybe then bounce again off other surfaces, and perhaps many more times before we hear them.

The longer paths mean that these reflected and scattered sounds arrive later. Moreover, each time they bounce some parts may be absorbed, and that absorption may vary with frequency, so the sound at our ears will have a different level and different spectrum from the original. All these overlapping, filtered, diminishing echoes contribute to our sense of the space.

Many, many different approaches have been used to simulate reverberation artificially. In the digital domain these fall into two categories: **algorithmic** (or **parametric**) reverb and **convolution** reverb.

An algorithmic reverb employs an abstracted model of how sound reverberates, usually combining a bunch of filtered delay lines. ([This walkthrough of writing a simple reverb effect](https://signalsmith-audio.co.uk/writing/2021/lets-write-a-reverb/) gives an interesting outline of what's involved.)

Here we apply an algorithmic reverb to our basic stereo loop:

In [ ]:
reverb = pedalboard.Reverb(room_size=0.75, damping=0.5, wet_level=0.5, dry_level=0.5)
play(reverb.process(PAN, SAMPLE_RATE))

Notice how different the sound is after multiple filtered delays than the original basic oscillations. The reverb doesn't just provide a sense of space, it significantly changes the timbres as well.

A convolution reverb uses the convolution operation introduced in week 4. As mentioned then, the physical process of a sound adding together with its own filtered and delayed echoes maps closely to convolution. How sounds propagate around a space can be recorded as an **impulse response**, and convolving a sound with that impulse response simulates how it would sound in the space.

Here are a couple of examples of impulse responses and their convolution with our loop.

In [ ]:
display(Audio('data/ir-1.wav'))
convo1 = pedalboard.Convolution('data/ir-1.wav')
play(convo1.process(PAN, SAMPLE_RATE))

In [ ]:
display(Audio('data/ir-2.wav'))
convo2 = pedalboard.Convolution('data/ir-2.wav')
play(convo2.process(PAN, SAMPLE_RATE))

Again, the loop sounds different in each case. Each impulse response picks out and combines different components of the timbre.

# Binaural Synthesis

A single room impulse response like the ones above measures how sound from a particular source location in the room propagates to a particular receiving location, capturing the reflective properties of the space.

We can use the same principle to measure how sound is heard at more than one point — specifically, at the two ears of a human listener. In this case we will often want to focus not on changes introduced by the room — ideally we would measure the responses in an anechoic chamber — but on those introduced by the listener's **head**. This will include the time and level differences we attempted to simulate earlier, and also some spectral filtering, especially of the higher frequencies. Convolving a sound with the impulse responses from each ear will produce a **binaural stereo** result that should sound well-localised when played back to the person (on headphones, so that each ear receives the appropriate signal in isolation, rather than muddied by additional head-related changes).

Since a sound will sound different depending on where it is relative to the listener, impulse responses must be recorded from very many different source locations. This is laborious and difficult. Moreover, impulse responses are specific to the individual — the shape of your head and ears will not be the same as mine.

It is rarely practical to record detailed **head-related impulse responses** (often referred to as **head-related transfer functions** or HRTFs, which are the same thing considered in the frequency domain) for every individual listener. Fortunately, there is a reasonable amount of anatomical similarity amongst humans, and so we can get at least part of the way by using **generic HRTFs**. These will not be exactly correct for any listener, but will often be close enough that the listener can get a decent approximation of spatialisation from the generated sound.

A number of generic HRTFs have been recorded and made available, and pyfar provides access to some of these. We can load a basic set like this:

In [ ]:
hrirs, sources = pf.signals.files.head_related_impulse_responses(
    position='horizontal', diffuse_field_compensation=True)

In this case we only have impulse responses for positions in a horizontal circle around the listener. (More sophisticated datasets will include sources above and below as well.)

In [ ]:
sources.show()

Suppose we want to simulate a sound occurring at a particular location, eg 90° to one side. We specify this in spherical coordinates. Azimuth denotes rotation about the vertical axis, with zero being directly to the front and the angle increasing to the left.



In [ ]:
elevation = 0
azimuth = np.pi/2
radius = 2

left_direction = pf.Coordinates.from_spherical_elevation(azimuth, elevation, radius)
right_direction = pf.Coordinates.from_spherical_elevation(-azimuth, elevation, radius)

We first need to find the appropriate impulse responses for our desired locations.

In [ ]:
left_index, _ = sources.find_nearest(left_direction)
right_index, _ = sources.find_nearest(right_direction)

# we can show the select source position on a plot
# it's not super visible in this case, but look for the red dots
sources.show(left_index)
sources.show(right_index)

We can plot the impulse responses to see what they look like:

In [ ]:
ax = pf.plot.time_freq(hrirs[right_index], label=['Left Ear', 'Right Ear'])
ax[0].legend()
ax[1].legend()

In [ ]:
ax = pf.plot.time_freq(hrirs[left_index], label=['Left Ear', 'Right Ear'])
ax[0].legend()
ax[1].legend()

Note that for sound on the right, the response at the left ear is at much lower amplitude than at the right ear, and a lot of the high frequencies are attenuated. And vice versa for sound on the left.

Let's try applying these IRs to some of our sounds.

In [ ]:
# convert to pyfar Signal objects
sig_kick = pf.Signal(KICK, SAMPLE_RATE)
sig_buzz = pf.Signal(BUZZ, SAMPLE_RATE)
sig_beep = pf.Signal(BEEP, SAMPLE_RATE)
sig_tink = pf.Signal(TINK, SAMPLE_RATE)
sig_kick_only = pf.Signal(KICK_ONLY, SAMPLE_RATE)
sig_hats_off = pf.Signal(HATS_OFF, SAMPLE_RATE)

In [ ]:
kick_left = pf.dsp.convolve(sig_kick, hrirs[left_index])
buzz_left = pf.dsp.convolve(sig_buzz, hrirs[left_index])
beep_right = pf.dsp.convolve(sig_beep, hrirs[right_index])
tink_right = pf.dsp.convolve(sig_tink, hrirs[right_index])

kick_only_left = pf.dsp.convolve(sig_kick_only, hrirs[left_index])
hats_off_right = pf.dsp.convolve(sig_hats_off, hrirs[right_index])

In [ ]:
pf.plot.time_freq(kick_left)
play(kick_left.time)

In [ ]:
pf.plot.time_freq(beep_right)
play(beep_right.time)

In [ ]:
pf.plot.time_freq(buzz_left)
play(buzz_left.time)

In [ ]:
pf.plot.time_freq(tink_right)
play(tink_right.time)

In [ ]:
pf.plot.time(kick_only_left + hats_off_right)
play((kick_only_left + hats_off_right).time)

We can simulate moving things around in space by changing the IRs we use over time. This can get a bit complicated and there are some practical issues about how to manage the switching, but here we'll just keep it simple, choosing one location per sound.

In [ ]:
def binaural_pan(sig, az, src=sources, irs=hrirs):
  """
  Wrapper for applying an HRIR to a signal to make it sound like
  it's coming from the specified source direction.
  """
  idx, _ = src.find_nearest(pf.Coordinates.from_spherical_elevation(az, 0, 2))
  return pf.dsp.convolve(sig, irs[idx])

In [ ]:
# build up a sequence of panned sounds
STEPS = 32
offsets = [np.pi/2, -np.pi/2, -np.pi/2, -np.pi/2]
sigs = [sig_kick, sig_tink, sig_tink, sig_tink]
angles = np.linspace(0, 2 * np.pi, STEPS)

segments = [ binaural_pan(sigs[ii % 4], angles[ii] + offsets[ii % 4]) for ii in range(STEPS) ]

# merge them all into a single track
collected = pf.utils.concatenate_channels(segments, caxis=1)

In [ ]:
pf.plot.time(collected)

The result is not perfect, but it does give a reasonable impression of the sounds moving around the listener.

In [ ]:
play(collected.time)

Finally, what happens if we combine the binaural head-related impulse response with a reverberant room response?

Convolution is linear, commutative and associative, so we can apply one after another and they should add up in a sensible way. We've made somewhat different assumptions in each case and so the spatial effects will not be perfectly consistent. But in practice it can still sound quite interesting:

In [ ]:
play(convo1.process(collected.time, SAMPLE_RATE))
play(convo2.process(collected.time, SAMPLE_RATE))